# Название проекта

# Предсказание возрастной категории пользователей интернет-сервисов

## Описание проекта

Заказчиком проекта является IT-компания «Йети», которая управляет группой популярных интернет-сервисов и использует рекламную сеть для показа контекстной рекламы. Один из важных факторов эффективности рекламы — корректный возрастной таргетинг. Если рекламное объявление показывается неподходящей возрастной группе, это может снижать конверсию, ухудшать пользовательский опыт и приводить к финансовым или репутационным рискам.

Цель проекта — построить модель машинного обучения, которая по данным о поведении анонимных пользователей в интернете будет предсказывать их возрастную категорию.

Задача относится к задаче многоклассовой классификации. Целевая переменная — `age_category`, принимающая значения:

- `0` — младше 18 лет;
- `1` — 18–25 лет;
- `2` — 26–40 лет;
- `3` — 41–55 лет;
- `4` — 56+ лет.

В качестве признаков используются данные из нескольких источников:

- информация о посещениях сайтов пользователями;
- активность взаимодействия с рекламными объявлениями;
- глубина просмотра сайтов;
- основное устройство пользователя;
- использование облачных сервисов.

В ходе проекта необходимо выполнить исследовательский анализ данных, обработать пропуски и дубликаты, сформировать единое признаковое пространство на уровне пользователей, обучить несколько моделей классификации и выбрать лучшую по качеству.

Основная метрика качества — `F1_macro`, так как важно одинаково хорошо предсказывать все возрастные категории, в том числе редкие классы. Дополнительно будут использоваться `precision_macro` и `recall_macro`.

Итоговая модель должна быть сохранена вместе с пайплайном предобработки и функцией формирования признаков, чтобы её можно было использовать для получения предсказаний на новых данных.

## Подготовка среды и библиотек

In [1]:
%%writefile requirements.txt
# Tested with Python 3.14.3
numpy==2.3.5
pandas==2.2.3
scipy==1.17.0
scikit-learn==1.8.0
matplotlib==3.10.8
seaborn==0.13.2
joblib==1.5.3
ipykernel==6.29.5

Overwriting requirements.txt


In [2]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, GridSearchCV
from sklearn.feature_selection import VarianceThreshold, RFE, SelectKBest, r_regression
from sklearn.svm import SVC
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

import joblib
from pathlib import Path

RANDOM_STATE = 42

In [3]:
REMOTE_DATA_URL = 'https://code.s3.yandex.net/datasets/'

file_names = {
    'users': 'ds_s13_users.csv',
    'visits': 'ds_s13_visits.csv',
    'ads_activity': 'ads_activity.csv',
    'surf_depth': 'surf_depth.csv',
    'primary_device': 'primary_device.csv',
    'cloud_usage': 'cloud_usage.csv'
}

dataframes = {}

for name, file_name in file_names.items():
    local_path = Path('datasets') / file_name          # локальная папка
    practicum_path = Path('/datasets') / file_name     # путь на платформе Практикума
    remote_path = REMOTE_DATA_URL + file_name          # удалённый путь
    
    try:
        dataframes[name] = pd.read_csv(local_path)
        print(f'{name}: загружен из локальной папки {local_path}')
    except FileNotFoundError:
        try:
            dataframes[name] = pd.read_csv(practicum_path)
            print(f'{name}: загружен из папки Практикума {practicum_path}')
        except FileNotFoundError:
            dataframes[name] = pd.read_csv(remote_path)
            print(f'{name}: загружен из удалённого хранилища')

users: загружен из локальной папки datasets\ds_s13_users.csv
visits: загружен из локальной папки datasets\ds_s13_visits.csv
ads_activity: загружен из локальной папки datasets\ads_activity.csv
surf_depth: загружен из локальной папки datasets\surf_depth.csv
primary_device: загружен из локальной папки datasets\primary_device.csv
cloud_usage: загружен из локальной папки datasets\cloud_usage.csv


In [8]:
df_users = dataframes['users']
df_visits = dataframes['visits']
df_ads_activity = dataframes['ads_activity']
df_surf_depth = dataframes['surf_depth']
df_primary_device = dataframes['primary_device']
df_cloud_usage = dataframes['cloud_usage']

df_mains = {
    'users': df_users,
    'visits': df_visits,
    'ads_activity': df_ads_activity,
    'surf_depth': df_surf_depth,
    'primary_device': df_primary_device,
    'cloud_usage': df_cloud_usage
}

In [9]:
def show_df_info(name, df):
    print('=' * 80)
    print(f'Датасет: {name}')
    print('=' * 80)
    
    print(f'Размер: {df.shape[0]} строк, {df.shape[1]} столбцов')
    
    print('\nИнформация о типах данных:')
    df.info()
    
    print('\nПервые 5 строк:')
    display(df.head())
    
    print('\nКоличество пропусков:')
    display(df.isna().sum().to_frame(name='missing_values'))
    
    print('\nКоличество явных дубликатов:')
    print(df.duplicated().sum())
    
    print('\n')

for name, df in df_mains.items():
    show_df_info(name, df)

Датасет: users
Размер: 5913 строк, 2 столбцов

Информация о типах данных:
<class 'pandas.DataFrame'>
RangeIndex: 5913 entries, 0 to 5912
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   user_id       5913 non-null   str  
 1   age_category  5913 non-null   int64
dtypes: int64(1), str(1)
memory usage: 92.5 KB

Первые 5 строк:


,user_id,age_category
0,f545-8c95aefe8d3e5548a689-a5b2fd39,4
1,cb48-5a0d6cde4d86ae10637e-c8ceb6ed,2
2,678b-614cd47d854b9d591db2-000b2e50,0
3,4ac0-dad169100b4a29b20818-b26ae7c5,4
4,f19b-9ac21ca973b41ecfa8c3-6a58191d,0



Количество пропусков:


,missing_values
user_id,0
age_category,0



Количество явных дубликатов:
87


Датасет: visits
Размер: 1065745 строк, 5 столбцов

Информация о типах данных:
<class 'pandas.DataFrame'>
RangeIndex: 1065745 entries, 0 to 1065744
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype
---  ------            --------------    -----
 0   date              1065745 non-null  str  
 1   daytime           1065745 non-null  str  
 2   session_id        1065745 non-null  str  
 3   user_id           1065745 non-null  str  
 4   website_category  1065745 non-null  str  
dtypes: str(5)
memory usage: 40.7 MB

Первые 5 строк:


...


Количество явных дубликатов:
0


Датасет: cloud_usage
Размер: 5680 строк, 2 столбцов

Информация о типах данных:
<class 'pandas.DataFrame'>
RangeIndex: 5680 entries, 0 to 5679
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      5680 non-null   str  
 1   cloud_usage  5680 non-null   bool 
dtypes: bool(1), str(1)
memory usage: 50.1 KB

Первые 5 строк:


,user_id,cloud_usage
0,a1e4-91c8a52eb855595e653f-298ce305,False
1,db9a-7b8e9e94448b7fcb19b6-4edca15f,False
2,0d55-9ad768879e9b08ca7ff9-843f76c7,True
3,4baa-43285d10a6d3cc969f2a-b21881d1,False
4,b8cd-cbb2411db005115ca64d-32700c62,False



Количество пропусков:


,missing_values
user_id,0
cloud_usage,0



Количество явных дубликатов:
0




## Исследовательский анализ данных

## Предобработка данных

## Обучение и оценка базовой модели

## Создание и отбор признаков

## Подбор гиперпараметров моделей

## Подготовка артефактов модели для внедрения

## Выводы о результатах работы